# 04 — Sistema Multiagente para Herdez Smart-Supply

En este notebook conectamos la salida del **Decision Engine** con una capa de agentes.

La idea no es que el LLM invente una decisión, sino que ayude a:

1. Interpretar alertas de riesgo.
2. Explicar costo-beneficio.
3. Contrastar escenarios.
4. Auditar supuestos.
5. Comunicar la decisión al Director de Supply Chain.

La arquitectura que vamos a defender es:

```text
XGBoost → Decision Engine determinista → CrewAI/LangChain/Gemini → Streamlit
```

## 1. Por qué primero hicimos el Decision Engine

Si conectáramos Gemini directamente al Excel y le preguntáramos *"¿qué hago?"*, el sistema podría sonar inteligente, pero sería débil técnicamente.

Por eso separamos responsabilidades:

| Capa | Responsable | Motivo |
|---|---|---|
| Predicción | XGBoost | Calcula riesgo de quiebre |
| Decisión financiera | Python determinista | Calcula pérdida, costo y beneficio |
| Validación | Reglas de negocio | Evita mover inventario incorrectamente |
| Explicación | LLM / agentes | Comunica y razona sobre resultados |

Frase para entrevista:

> "El LLM no toma decisiones financieras libres; interpreta resultados auditables generados por modelos y reglas deterministas."

## 2. Instalación de dependencias

En Colab o local puedes instalar las librerías de agentes. Si no tienes API key, el script igual funciona en modo fallback determinista.

In [ ]:
# En Colab descomenta si hace falta:
# !pip install crewai langchain langchain-google-genai google-generativeai

## 3. Configurar API Key de Gemini

Puedes usar `GEMINI_API_KEY` o `GOOGLE_API_KEY`.

En Colab, una forma simple para demo es:

In [ ]:
# import os
# os.environ["GEMINI_API_KEY"] = "TU_API_KEY_AQUI"

## 4. Ejecutar el sistema agente

El modo `auto` intenta:

1. CrewAI multiagente.
2. LangChain + Gemini como fallback intermedio.
3. Fallback determinista si no hay API key o librerías.

Esto es una decisión de arquitectura importante: **la demo no debe morir porque falle un proveedor externo**.

In [ ]:
!python ../src/04_agent_system.py   --recommendations ../outputs/decision_recommendations.csv   --scenarios ../outputs/decision_transfer_scenarios.csv   --output-dir ../outputs   --mode auto   --max-rows-context 8

## 5. Leer el brief ejecutivo generado

Este archivo es el que después puede mostrarse en Streamlit o usarse como respuesta del agente.

In [ ]:
from pathlib import Path
report_path = Path("../outputs/agent_executive_brief.md")
print(report_path.read_text(encoding="utf-8")[:4000])

## 6. Qué agentes estamos modelando

Aunque el fallback determinista no llama a Gemini, el diseño real multiagente es este:

| Agente | Función |
|---|---|
| Inventory Risk Analyst | Interpreta la predicción de riesgo |
| Logistics Cost Analyst | Analiza pérdida esperada, costo y beneficio |
| Supply Chain Strategist | Construye plan operativo |
| Policy Validator / Critic | Revisa riesgos y guardrails |
| Executive Communicator | Explica al director y al gerente técnico |

Esto demuestra creatividad sin sacrificar control técnico.

## 7. Por qué no enviamos todo el dataset al LLM

Mandar todo el CSV al LLM sería mala práctica por tres razones:

1. Más costo y tokens.
2. Más ruido.
3. Más riesgo de que el modelo invente relaciones.

En vez de eso, enviamos un contexto priorizado:

```text
Top alertas + métricas calculadas + explicación de arquitectura
```

Esto es más profesional porque el LLM trabaja sobre evidencia resumida y auditada.

## 8. Cómo defender CrewAI + LangChain + Gemini

Argumento recomendado:

> "Usé CrewAI para coordinar agentes especializados, LangChain como capa de abstracción de modelo y Gemini Flash como motor de razonamiento. Sin embargo, la decisión crítica se mantiene fuera del LLM: XGBoost predice riesgo y Python calcula costo-beneficio. Así mantengo portabilidad, bajo costo de prototipo y una ruta clara hacia GCP."

## 9. Siguiente paso

Después de este archivo, el siguiente componente es el dashboard:

```text
05_streamlit_dashboard.py
```

Ahí vamos a mostrar:

- KPIs ejecutivos.
- Alertas priorizadas.
- Recomendaciones.
- Brief generado por el agente.
- Gráficos de riesgo por SKU/CEDI.
- Simulador de decisión.